In [ ]:
# Data: ../csv/Fig1_E_MUA_d2_correlation_FigS1_G_plotting_dynrange/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'Fig1_E_MUA_d2_correlation_FigS1_G_plotting_dynrange'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
import os, re, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
# ---------- Summary stats ----------

# ---------- 1) Load all pickles ----------
# ---- data (../csv) ----
all_pairs = read_csv('d2_mua_pairs_18sessions.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# FOLDER = r"D:\Documents\GitHub\Sleep_criticaltiy\data\cross_state\pairs"
# files = sorted(glob.glob(os.path.join(FOLDER, "*_d2_mua_pairs.pkl")))
# assert files, f"No pairs .pkl files in {FOLDER}"
# 
# dfs = []
# for f in files:
#     try:
#         df = pd.read_pickle(f)
#         # ensure keys exist; infer from filename if needed
#         if "animal" not in df.columns or "day" not in df.columns or "region" not in df.columns:
#             base = os.path.basename(f)
#             m = re.match(r"([^_]+)_day([^_]+)_([^_]+)_d2_mua_pairs\.pkl", base)
#             if m:
#                 df["animal"] = df.get("animal", m.group(1))
#                 df["day"]    = df.get("day",    m.group(2))
#                 df["region"] = df.get("region", m.group(3))
#             else:
#                 df["animal"] = df.get("animal", "NA")
#                 df["day"]    = df.get("day", "NA")
#                 df["region"] = df.get("region", "NA")
#         if "animal_day" not in df.columns:
#             df["animal_day"] = df["animal"].astype(str) + "_" + df["day"].astype(str)
#         dfs.append(df)
#     except Exception as e:
#         print(f"⚠️ Skipping {os.path.basename(f)}: {e}")
# 
# all_pairs = pd.concat(dfs, ignore_index=True)
# all_pairs["group"] = all_pairs["group"].astype(str)
# 
# from pathlib import Path
# import numpy as np
# import pandas as pd
# from neuro_py.io import loading
# # root of the PPP data
# PPP_ROOT = Path(r"W:\data\PPP")
# 
# # we will add this new column
# all_pairs = all_pairs.copy()
# all_pairs["epoch_name"] = np.nan
# 
# # only care about wake θ / wake non-θ
# wake_mask = all_pairs["group"].isin(["Wake θ", "Wake non-θ"])
# 
# # loop over each (animal, day)
# for (animal, day), idx_sub in all_pairs[wake_mask].groupby(["animal", "day"]).groups.items():
#     idx_sub = np.asarray(idx_sub)
#     sub = all_pairs.loc[idx_sub]
# 
#     basepath = PPP_ROOT / animal / day   # e.g. W:\data\PPP\PPP20\day4
#     print(f"Annotating {animal} {day}  ->  {basepath}")
# 
#     # load epoch_df for this basepath
#     epoch_df = loading.load_epoch(str(basepath))
# 
#     # --- pick T-maze / task epochs, excluding anything that looks like sleep ---
#     name = epoch_df["name"].astype(str).str.lower()
#     env  = epoch_df.get("environment", pd.Series("", index=epoch_df.index)).astype(str).str.lower()
# 
#     is_sleep = env.str.contains("sleep") | name.str.contains("sleep")
# 
#     # "T_", "Tmaze", "T_maze" etc. adjust patterns as needed
#     is_tmaze = (
#         env.str.contains("t_maze") | env.str.contains("tmaze") | env.str.contains("t-maze") | env.str.contains("task") |
#         name.str.contains("t_maze") | name.str.contains("tmaze") | name.str.contains("t-maze") |
#         name.str.startswith("freet") | name.str.startswith("t_")      # freeT, T_...
#     )
# 
#     task_epochs = epoch_df[is_tmaze & ~is_sleep].copy()
#     if task_epochs.empty:
#         print(f"  -> no T-maze / task epochs found, skipping.")
#         continue
# 
#     # for each task epoch, tag points whose time falls inside [startTime, stopTime]
#     times = sub["time"].to_numpy(float)
# 
#     for _, row in task_epochs.iterrows():
#         t0 = float(row["startTime"])
#         t1 = float(row["stopTime"])
#         ep_name = row["name"]
# 
#         in_epoch = (times >= t0) & (times <= t1)
#         if not in_epoch.any():
#             continue
# 
#         # write epoch name back into all_pairs
#         all_pairs.loc[idx_sub[in_epoch], "epoch_name"] = ep_name

# wake θ / non-θ in T-maze epochs, selected as:
theta_task     = all_pairs[(all_pairs["group"] == "Wake θ") &
                           all_pairs["epoch_name"].notna()]
non_theta_task = all_pairs[(all_pairs["group"] == "Wake non-θ") &
                           all_pairs["epoch_name"].notna()]

In [ ]:
# assume all_pairs already has:
# - 'd2'
# - 'MUA'  (or rename from 'mua30' if needed)
# - 'group'         (NREM, REM, Wake θ, etc.)
# - 'epoch_name'    (NaN for non-task, string for T-maze epochs)
# - 'animal', 'day' (for per-file ID)
# --- Tile by d2_z per state (6 tiles) ---
# ---------- Summary stats ----------
import matplotlib as mpl
# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
# -------- settings --------
N_BINS = 6
MIN_PER_BIN = 2
BASELINE_Q = 0.25
STATE_LIST = ["NREM", "WAKE"]

# ---------- utils ----------
def coerce_to_df(dfs):
    if isinstance(dfs, pd.DataFrame):
        df = dfs.copy()
        if "__file__" not in df.columns:
            df["__file__"] = "file1"
        return df
    if isinstance(dfs, dict):
        parts = []
        for name, d in dfs.items():
            if isinstance(d, pd.DataFrame):
                t = d.copy()
                t["__file__"] = str(name)
                parts.append(t)
        return pd.concat(parts, ignore_index=True)
    parts = []
    for i, d in enumerate(dfs, start=1):
        if isinstance(d, pd.DataFrame):
            t = d.copy()
            if "__file__" not in t.columns:
                t["__file__"] = f"file{i}"
            parts.append(t)
    return pd.concat(parts, ignore_index=True)

def plot_bars(dfB, ycol, xlabel, title):
    if dfB.empty:
        return
    x = np.arange(len(dfB))
    plt.figure(figsize=(6.5,4))
    plt.bar(x, dfB[ycol].values)
    plt.xticks(x, [f"{v:.2g}" for v in dfB["d2_center"].values], rotation=45, ha="right")
    plt.ylabel(ycol)
    plt.xlabel(xlabel)
    plt.title(title)
    plt.tight_layout()
    plt.show()

# ---------- per-file d2 normalization ----------
def add_perfile_d2_normalized(df, method="percentile"):
    """
    Adds a normalized d2 column 'd2n' computed per file.
    method: 'percentile' (0..1), 'z', or 'minmax'
    """
    df = df.copy()
    def _percentile_col(g):
        x = g["d2"].to_numpy(float)
        # ranks in [0,1]
        order = np.argsort(x)
        ranks = np.empty_like(order, dtype=float)
        ranks[order] = np.arange(x.size, dtype=float)
        q = ranks / max(x.size - 1, 1)  # avoid div/0 for size=1
        g["d2n"] = q
        return g

    def _z_col(g):
        # clip to 1–99% to compute robust-ish mean/std
        lo, hi = g["d2"].quantile([0.01, 0.99])
        xc = g["d2"].clip(lo, hi)
        mu, sd = xc.mean(), xc.std(ddof=0)
        sd = sd if sd > 0 else np.nan
        g["d2n"] = (g["d2"] - mu) / sd
        return g

    def _minmax_col(g):
        lo, hi = g["d2"].min(), g["d2"].max()
        span = hi - lo
        g["d2n"] = (g["d2"] - lo) / (span if span > 0 else np.nan)
        return g

    if method == "percentile":
        df = df.groupby("__file__", group_keys=False).apply(_percentile_col)
    elif method == "z":
        df = df.groupby("__file__", group_keys=False).apply(_z_col)
    elif method == "minmax":
        df = df.groupby("__file__", group_keys=False).apply(_minmax_col)
    else:
        raise ValueError("method must be 'percentile', 'z', or 'minmax'")
    return df

def perfile_dyn_by_bin(dstate, method="percentile", bin_mode="equalwidth"):
    """
    Returns one row per (__file__, d2_bin) with per-file dynamic-range metric.
    
    method:
        'percentile', 'z', 'minmax'  -> how to normalize d2 per file into 'd2n'
    bin_mode:
        'equalwidth' -> fixed-width bins in normalized d2n
        'tile'       -> quantile / equal-count bins (forced tiles)
    """
    if dstate.empty:
        return pd.DataFrame(columns=["__file__", "d2_bin", "d2_center", "dyn", "n_in_bin"])

    # 1) normalize d2 per file
    dstate = add_perfile_d2_normalized(dstate, method=method)

    # keep only finite d2n
    dstate = dstate[np.isfinite(dstate["d2n"])].copy()
    if dstate.empty:
        return pd.DataFrame(columns=["__file__", "d2_bin", "d2_center", "dyn", "n_in_bin"])

    # 2) define bins
    if bin_mode == "equalwidth":
        if method in ("percentile", "minmax"):
            bins = np.linspace(0, 1, N_BINS+1)
            centers = (bins[:-1] + bins[1:]) / 2.0
        else:  # z
            zmin, zmax = dstate["d2n"].quantile([0.01, 0.99])
            if not np.isfinite(zmin) or not np.isfinite(zmax) or zmax <= zmin:
                return pd.DataFrame(columns=["__file__", "d2_bin", "d2_center", "dyn", "n_in_bin"])
            bins = np.linspace(zmin, zmax, N_BINS+1)
            centers = (bins[:-1] + bins[1:]) / 2.0

        dstate["d2_bin"] = pd.cut(dstate["d2n"], bins=bins, labels=False, include_lowest=True)
        # lookup centers by integer bin index
        center_lookup = {i: float(c) for i, c in enumerate(centers)}

    elif bin_mode == "tile":
        # forced tiles: equal-count quantile bins across all rows
        # labels are 0..k-1; duplicates="drop" may give < N_BINS if data are degenerate
        dstate["d2_bin"] = pd.qcut(
            dstate["d2n"],
            q=N_BINS,
            labels=False,
            duplicates="drop"
        )
        # compute center as the median d2n within each tile
        centers_ser = (
            dstate
            .groupby("d2_bin")["d2n"]
            .median()
            .dropna()
        )
        center_lookup = centers_ser.to_dict()
    else:
        raise ValueError("bin_mode must be 'equalwidth' or 'tile'")

    # 3) per-(file, bin) mean of raw MUA
    fb_mean = dstate.groupby(["__file__", "d2_bin"])["MUA"].transform("mean")

    # normalize: mean within each (file, bin) becomes 1
    eps = 1e-12
    dstate["MUA_norm"] = dstate["MUA"] / (fb_mean.replace(0, np.nan) + eps)

    # keep positive finite values
    dstate = dstate[np.isfinite(dstate["MUA_norm"]) & (dstate["MUA_norm"] > 0)]

    rows = []
    for (f, b), g in dstate.groupby(["__file__", "d2_bin"], sort=True):
        if pd.isna(b):
            continue
        x = g["MUA_norm"].to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < MIN_PER_BIN:
            continue

        p05, p95 = np.percentile(x, [5, 95])
        b_int = int(b)
        if b_int not in center_lookup:
            continue

        rows.append({
            "__file__": f,
            "d2_bin": b_int,
            "d2_center": float(center_lookup[b_int]),
            "dyn": float(10 * np.log10(p95 / p05)),
            "n_in_bin": int(x.size),
        })
    return pd.DataFrame(rows)


def summarize_mean_sem(df_perfile):
    """
    Aggregates over files: mean and SEM per d2_bin.
    """
    if df_perfile.empty:
        return pd.DataFrame(columns=["d2_bin", "d2_center", "mean", "sem", "n_files"])
    g = df_perfile.groupby("d2_bin", as_index=False)
    out = g.agg(
        d2_center=("d2_center", "median"),
        mean=("dyn", "mean"),
        std=("dyn", "std"),
        n_files=("__file__", "nunique"),
    )
    out["sem"] = out["std"] / np.sqrt(out["n_files"].clip(lower=1))
    return out[["d2_bin", "d2_center", "mean", "sem", "n_files"]].sort_values("d2_center")

def plot_mean_sem(dfS, xlabel, title, color, ylabel="dynamic range (p95–p05)", save_path=None):
    if dfS.empty:
        return
    
    y_data = dfS["d2_center"].values
    x_data = dfS["mean"].values
    x_error = dfS["sem"].values

    plt.figure(figsize=(4, 6))

    # Plot horizontally: d2 bin center on y, mean on x
    plt.errorbar(x_data, y_data, xerr=x_error, color=color,fmt='-o', capsize=4, linewidth=2)

    plt.xlabel(ylabel)
    plt.ylabel(xlabel)
    
    plt.title(title)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    plt.show()

method = "minmax"  # or 'percentile' or 'z'
xlab = ("per-file d² percentile (bin center)" if method in ("percentile","minmax")
        else "per-file d² z-score (bin center)")

from scipy.stats import spearmanr, norm
import numpy as np, pandas as pd
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm

def spearman_stouffer(T, alternative="increasing", min_unique=3, weight_mode="sqrt(df)"):
    """
    Per-file Spearman correlation between d2_bin and dyn, then Stouffer combine signed z.

    Numerical safeguards:
      - clip p2 away from {0,1} before norm.isf
      - skip nan rho/p
      - skip degenerate groups (too few unique bins / constant arrays)
      - ensure combined z uses only finite z and positive weights

    alternative:
      - "increasing": one-sided p = P(Z >= z_comb) = sf(z_comb)
      - "decreasing": one-sided p = P(Z <= z_comb) = cdf(z_comb)
    """
    rows = []

    # smallest positive float64 (subnormal); avoids p==0 issues
    pmin = np.nextafter(0.0, 1.0)
    pmax = 1.0 - pmin

    for f, g in T.groupby("__file__"):
        n_unique = g["d2_bin"].nunique(dropna=True)
        if n_unique < min_unique:
            continue

        x = np.asarray(g["d2_bin"], float)
        y = np.asarray(g["dyn"], float)

        # drop NaNs pairwise
        m = np.isfinite(x) & np.isfinite(y)
        x = x[m]; y = y[m]
        if x.size < 3:
            continue
        # if either is constant, spearmanr can return nan
        if np.all(x == x[0]) or np.all(y == y[0]):
            continue

        rho, p2 = spearmanr(x, y)  # two-sided p
        if (rho is None) or (p2 is None) or (not np.isfinite(rho)) or (not np.isfinite(p2)):
            continue

        # Clip p2 to avoid p2/2 == 0 -> isf(0)=inf
        p2 = float(np.clip(p2, pmin, pmax))

        # Convert two-sided p to |z| then apply sign of rho
        # p(|Z| >= |z|) = p2  =>  p(Z >= |z|) = p2/2  =>  |z| = isf(p2/2)
        abs_z = norm.isf(0.5 * p2)  # stable because p2 clipped
        z = np.copysign(abs_z, rho)

        # weight choice
        if weight_mode == "sqrt(df)":
            # df ~ n_unique - 2
            w = np.sqrt(max(n_unique - 2, 1))
        elif weight_mode == "sqrt(n)":
            w = np.sqrt(x.size)
        else:
            w = 1.0

        if np.isfinite(z) and np.isfinite(w) and w > 0:
            rows.append({"__file__": f, "rho": rho, "p2": p2, "z": z, "w": w, "n_unique": n_unique, "n": x.size})

    R = pd.DataFrame(rows)
    if R.empty:
        return None

    # Combine only finite entries (extra safety)
    mask = np.isfinite(R["z"].to_numpy()) & np.isfinite(R["w"].to_numpy()) & (R["w"].to_numpy() > 0)
    R = R.loc[mask].copy()
    if R.empty:
        return None

    w = R["w"].to_numpy()
    z = R["z"].to_numpy()

    denom = np.sqrt(np.sum(w**2))
    if denom == 0 or not np.isfinite(denom):
        return None

    z_comb = float(np.sum(w * z) / denom)

    # Stable tail probabilities (avoid 1 - cdf)
    alt = alternative.lower()
    if alt in ("increasing", "greater", "pos", "+"):
        p1 = float(norm.sf(z_comb))   # P(Z >= z_comb)
    elif alt in ("decreasing", "less", "neg", "-"):
        p1 = float(norm.cdf(z_comb))  # P(Z <= z_comb)
    else:
        raise ValueError(f"Unknown alternative={alternative!r}. Use 'increasing' or 'decreasing'.")
        # Summaries of per-file Spearman r (rho)
    rho = R["rho"].to_numpy(float)
    w   = R["w"].to_numpy(float)

    rho_clip = np.clip(rho, -0.999999, 0.999999)
    z_rho = np.arctanh(rho_clip)                 # Fisher transform
    z_bar = np.sum(w * z_rho) / np.sum(w)        # weighted mean in z-space
    rho_wmean_fisher = float(np.tanh(z_bar))     # back-transform
    rho_median = float(np.median(rho))


    # Optional: also return a reporting-safe string floor (doesn't change p1)
    return {
        "z_comb": z_comb,
        "p_one_sided": p1,
        "n_files": len(R),
        "rho_wmean_fisher": rho_wmean_fisher,
        "rho_median": rho_median,
        "per_file": R
    }


# build a tidy table: one row per (__file__, d2_bin) with dyn and bin center
def perfile_dyn_by_bin_no_baseline(dstate, method="minmax"):
    dstate = add_perfile_d2_normalized(dstate, method=method).copy()
    # bins in normalized d2 space
    if method in ("percentile","minmax"):
        bins = np.linspace(0, 1, N_BINS+1)
    else:
        zmin, zmax = dstate["d2n"].quantile([0.01, 0.99])
        bins = np.linspace(zmin, zmax, N_BINS+1)
    centers = (bins[:-1] + bins[1:]) / 2.0

    dstate = dstate[np.isfinite(dstate["d2n"])].copy()
    dstate["d2_bin"] = pd.cut(dstate["d2n"], bins=bins, labels=False, include_lowest=True)

    # per-(file,bin) mean-normalized MUA
    fb_mean = dstate.groupby(["__file__", "d2_bin"])["MUA"].transform("mean")
    dstate["MUA_norm"] = dstate["MUA"] / fb_mean.clip(lower=1e-12)

    rows = []
    for (f, b), g in dstate.groupby(["__file__", "d2_bin"], sort=True):
        if pd.isna(b): 
            continue
        x = g["MUA_norm"].to_numpy(float)
        x = x[np.isfinite(x) & (x > 0)]
        if x.size < MIN_PER_BIN:
            continue
        p05, p95 = np.percentile(x, [5, 95])
        rows.append({
            "__file__": f,
            "d2_bin": int(b),
            "d2_center": float(centers[int(b)]),
            "dyn": float(10 * np.log10(p95 / p05)),
            "n_in_bin": int(x.size),
        })
    return pd.DataFrame(rows)

df_base = all_pairs.copy()

# ensure we have a per-file identifier
if "__file__" not in df_base.columns:
    df_base["__file__"] = df_base["animal"].astype(str) + "_" + df_base["day"].astype(str)

# ---- pick the MUA column name ----
mua_col = None
for cand in ["MUA", "mua30", "mua", "MUA30"]:
    if cand in df_base.columns:
        mua_col = cand
        break

if mua_col is None:
    raise KeyError("Could not find an MUA column in all_pairs. Looked for: 'MUA', 'mua30', 'mua', 'MUA30'.")


# normalize name to 'MUA' for the DR pipeline
df_base = df_base.rename(columns={mua_col: "MUA"})

# -------------------------
# define Sleep and Task-wake
# -------------------------
sleep_set     = {"NREM", "REM", "REM4", "Sleep", "SLEEP"}
wake_task_set = {"Wake θ", "Wake non-θ", "Wake", "WAKE"}

# Sleep = all NREM/REM pooled
sleep_mask = df_base["group"].isin(sleep_set)
df_sleep = df_base.loc[sleep_mask, ["d2", "MUA", "__file__"]].dropna()

# Task-wake = wake + inside T-maze epochs (epoch_name not NaN)
task_mask = (
    df_base["group"].isin(wake_task_set) &
    df_base["epoch_name"].notna()
)
df_taskwake = df_base.loc[task_mask, ["d2", "MUA", "__file__"]].dropna()


method = "minmax"   # or 'percentile' / 'z' as before

colors_cond = {
    "Sleep": "#7b3294",
    "Task-wake": "#d95f0e",
}

conditions = [
    ("Sleep", df_sleep),
    ("Task-wake", df_taskwake),
]

for label, dstate in conditions:
    if dstate.empty:
        continue

    # --- dynamic range per file, per d2 bin (with normalization) ---
    df_perfile = perfile_dyn_by_bin(dstate, method=method)
    df_summary = summarize_mean_sem(df_perfile)


    plot_mean_sem(
        df_summary,
        xlabel="Normalized d² distance",
        title=label,
        color=colors_cond[label],
        ylabel="normalized MUA dynamic range",
        save_path=None
    )

    # --- monotonicity test via Spearman + Stouffer meta ---
    T = perfile_dyn_by_bin_no_baseline(dstate, method="minmax")
    res = spearman_stouffer(T, alternative="decreasing")  # dyn decreasing with d2?
    if res is not None:

        # Optional: show per-file rhos (top/bottom)
        R = res["per_file"].sort_values("rho")
    else:
        pass

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =======================================
# 1) Prep df_base and MUA column
# =======================================

df_base = all_pairs.copy()

# per-file identifier
if "__file__" not in df_base.columns:
    df_base["__file__"] = df_base["animal"].astype(str) + "_" + df_base["day"].astype(str)

# pick the MUA column name
mua_col = None
for cand in ["MUA", "mua30", "mua", "MUA30"]:
    if cand in df_base.columns:
        mua_col = cand
        break

if mua_col is None:
    raise KeyError(
        "Could not find an MUA column in all_pairs. "
        "Looked for: 'MUA', 'mua30', 'mua', 'MUA30'."
    )

df_base = df_base.rename(columns={mua_col: "MUA"})

# =======================================
# 2) State definitions & colors
# =======================================

state_label_sets = {
    "NREM":      {"NREM"},
    "REM":       {"REM", "REM4"},
    "THETA":     {"THETA", "Wake θ"},
    "non-THETA": {"nonTHETA", "Wake non-θ"},
}

method = "minmax"   # or 'percentile' / 'z'

colors_cond = {
    "NREM":      "#7b3294",
    "REM":       "#d7b5d8",
    "THETA":     "#d95f0e",
    "non-THETA": "#fdb863",
}

# raw summaries (for printing/stats) and per-state normalized summaries (for plotting)
summaries_raw  = {}
summaries_norm = {}

# =======================================
# 3) Per-state summaries + monotonicity
# =======================================
def p_to_stars(p):
    p = float(p)
    if not np.isfinite(p):
        return ""
    if p < 1e-4:
        return "****"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "ns"

for label, groups in state_label_sets.items():
    mask = df_base["group"].isin(groups)
    dstate = df_base.loc[mask, ["d2", "MUA", "__file__"]].dropna()

    if dstate.empty:
        continue

    # --- dynamic range per file, per d2 bin ---
    df_perfile = perfile_dyn_by_bin(dstate, method=method)
    df_summary = summarize_mean_sem(df_perfile)
    summaries_raw[label] = df_summary


    # --- per-state 0–1 normalization just for plotting ---
    if not df_summary.empty:
        dfN = df_summary.copy()
        m = dfN["mean"].to_numpy()
        mn = np.nanmin(m)
        mx = np.nanmax(m)

        if np.isfinite(mn) and np.isfinite(mx) and not np.isclose(mn, mx):
            dfN["mean"] = (dfN["mean"] - mn) / (mx - mn)
            dfN["sem"]  = dfN["sem"] / (mx - mn)
        else:
            # degenerate case: all values equal → set mean=0.5, sem=0
            dfN["mean"] = 0.5
            dfN["sem"]  = 0.0

        summaries_norm[label] = dfN
    else:
        summaries_norm[label] = df_summary

    # --- monotonicity test via Spearman + Stouffer meta (use RAW, not normalized) ---
    T = perfile_dyn_by_bin_no_baseline(dstate, method=method)
    res = spearman_stouffer(T, alternative="decreasing")  # dyn decreasing with d2?
    if res is not None:
        stars = p_to_stars(res["p_one_sided"])
        print(
            f"{label}: "
            f"rho_wmean_fisher = {res['rho_wmean_fisher']:+.3f}, p_one_sided = {res['p_one_sided']:.3e}, "
            f"files = {res['n_files']}"
        )

        # Optional: show per-file rhos (top/bottom)
        R = res["per_file"].sort_values("rho")
    else:
        print(f"{label}: not enough bins/files for meta-analysis.")

# =======================================
# 4) Combined plots using per-state normalized dynamic range
# =======================================

# --- (A) NREM vs REM ---

plt.figure(figsize=(4, 6))
ax = plt.gca()

for label in ["NREM", "REM"]:
    if label not in summaries_norm:
        continue
    dfS = summaries_norm[label]
    if dfS.empty:
        continue

    # x-axis: per-state normalized (or z-scored) mean dynamic range
    x = dfS["mean"].to_numpy(float)
    e = dfS["sem"].to_numpy(float)

    # y-axis: bin index 1..6
    if "d2_bin" in dfS.columns:
        y = dfS["d2_bin"].to_numpy(int) + 1   # if 0-based
    elif "tile_num" in dfS.columns:
        y = dfS["tile_num"].to_numpy(int)     # if already 1..6
    else:
        raise KeyError("Need a bin column like 'd2_bin' or 'tile_num' in dfS to plot y=1..6.")

    ax.errorbar(
        x, y,
        xerr=e,
        fmt='-o',
        capsize=4,
        linewidth=2,
        label=label,
        color=colors_cond.get(label, "k"),
    )

# y-axis formatting for 6 bins
ax.set_ylim(0.5, 6.5)

ax.set_xlabel("normalized MUA dynamic range")
ax.set_ylabel("d² bin")
ax.set_title("Dynamic range vs d² (NREM / REM)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
plt.tight_layout()

save_path = rf"../figure_output\fig1_E_NREM_vs_REM_dyn_by_d2_{method}_norm.svg"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()


# --- (B) THETA vs non-THETA ---
plt.figure(figsize=(4, 6))
ax = plt.gca()

for label in ["THETA", "non-THETA"]:
    if label not in summaries_norm:
        continue
    dfS = summaries_norm[label]
    if dfS.empty:
        continue

    # x-axis: per-state normalized (or z-scored) mean dynamic range
    x = dfS["mean"].to_numpy(float)
    e = dfS["sem"].to_numpy(float)

    # y-axis: bin index 1..6
    if "d2_bin" in dfS.columns:
        y = dfS["d2_bin"].to_numpy(int) + 1   # if 0-based
    elif "tile_num" in dfS.columns:
        y = dfS["tile_num"].to_numpy(int)     # if already 1..6
    else:
        raise KeyError("Need a bin column like 'd2_bin' or 'tile_num' in dfS to plot y=1..6.")

    ax.errorbar(
        x, y,
        xerr=e,
        fmt='-o',
        capsize=4,
        linewidth=2,
        label=label,
        color=colors_cond.get(label, "k"),
    )

ax.set_xlabel("normalized MUA dynamic range ")
ax.set_ylabel("normalized d² distance")
ax.set_title("Dynamic range vs d² (THETA / non-THETA)")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
plt.tight_layout()
save_path = rf"../figure_output\fig1_E_THETA_vs_nonTHETA_dyn_by_d2_{method}_norm.svg"
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
from matplotlib.lines import Line2D
from scipy import stats
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# 1) Collapse to Wake vs Sleep
# -----------------------------
wake_set  = {"Wake θ","Wake non-θ","Wake","WAKE"}
sleep_set = {"NREM","REM","REM4","Sleep","SLEEP"}

all_pairs = all_pairs.copy()
all_pairs["group2"] = all_pairs["group"].map(
    lambda g: "Wake" if g in wake_set else ("Sleep" if g in sleep_set else np.nan)
)
pairs_ws = all_pairs.dropna(subset=["group2"]).copy()

# -----------------------------
# 2) z-score within (animal_day, group2)
# -----------------------------
def zscore_within(df, by=("animal_day", "group2")):
    out = df.copy()

    def safe_z(s):
        s = s.astype(float)
        sd = s.std(ddof=0)
        if (not np.isfinite(sd)) or sd == 0:
            return pd.Series(np.nan, index=s.index)
        return (s - s.mean()) / sd

    out["d2_z"]    = out.groupby(list(by), dropna=False)["d2"].transform(safe_z)
    out["mua30_z"] = out.groupby(list(by), dropna=False)["mua30"].transform(safe_z)
    return out

z_pairs = zscore_within(pairs_ws, by=("animal_day", "group2"))

# --------------------------------
# 3) Define task wake & sleep sets
# --------------------------------

# --- Task wake: epochs with a T-maze label & wake θ / non-θ ---
task_wake_mask = (
    z_pairs["epoch_name"].notna() &
    z_pairs["group"].isin(["Wake θ", "Wake non-θ"])
)
task_wake = z_pairs[task_wake_mask].copy()

# --- Sleep: NREM vs REM/REM4 ---
def map_sleep(g):
    if g in {"NREM"}:
        return "NREM"
    if g in {"REM", "REM4"}:
        return "REM"
    return np.nan

z_pairs["sleep_substate"] = z_pairs["group"].map(map_sleep)
sleep_mask = z_pairs["sleep_substate"].notna()
sleep_pairs = z_pairs[sleep_mask].copy()
# --------------------------------
# 5) Helper: add regression line
# --------------------------------
def add_fit_line(ax, x, y, x_min, x_max, *, color='k', linestyle='--', lw=1.6):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return None, np.nan, np.nan
    lr = stats.linregress(x[ok], y[ok])
    xs = np.linspace(x_min, x_max, 200)
    ax.plot(xs, lr.slope * xs + lr.intercept,
            color=color, linestyle=linestyle, linewidth=lw)
    r, p = stats.pearsonr(x[ok], y[ok])
    return lr, r, p

# --------------------------------
# 6) Palettes
# --------------------------------
colors = {
    "NREM": "#56236a",
    "REM": "#976695",
    "THETA": "#97550a",
    "nonTHETA": "#b87f44"

}
wake_palette  = {"Wake θ": "#d95f0e", "Wake non-θ": "#fdb863"}
sleep_palette = {"NREM": "#7b3294", "REM": "#d7b5d8"}
wake_inline_palette  = {"Wake θ": "#97550a", "Wake non-θ": "#b87f44"}
sleep_inline_palette = {"NREM": "#56236a", "REM": "#976695"}
fig, axes = plt.subplots(1, 2, figsize=(8.5, 5), sharex=False, sharey=False)
import numpy as np
import mpmath as mp

def pfmt_extreme_p(p, sig=3):
    """
    Format a p-value without underflow.
    - If p is representable as float64 -> scientific notation
    - If p underflowed to 0.0 -> return '< 5e-324'
    - If p is given as a *log p* elsewhere, use the log version instead.
    """
    # handle None / nan
    if p is None:
        return "nan"
    p = float(p)
    if not np.isfinite(p):
        return "nan"
    if p == 0.0:
        return "< 5e-324"

    # normal case
    return f"{p:.{sig}e}"

# --------------------------------
# LEFT: task wake (θ vs non-θ)
# --------------------------------
ax = axes[0]
legend_handles = []
def local_limits(x, pad_frac=0.1):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return -1, 1
    lo, hi = np.percentile(x, [1, 99])
    span = hi - lo if hi > lo else 1.0
    pad = pad_frac * span
    return float(lo - pad), float(hi + pad)

for st in ["Wake non-θ", "Wake θ"]:
    sub = task_wake[task_wake["group"] == st]
    if sub.empty:
        continue

    x = sub["mua30_z"].to_numpy(float)
    y = sub["d2_z"].to_numpy(float)

    # scatter points
    sc = ax.scatter(
        x, y,
        s=4, alpha=0.8,
        color=wake_palette[st],
        edgecolors="none",
        rasterized=True
    )
    sc.set_rasterized(True)
    x_min_state, x_max_state = local_limits(x)
    # fit line for THIS state only
    lr, r, p = add_fit_line(
        ax, x, y,
        x_min=x_min_state, x_max=x_max_state,
        color=wake_inline_palette[st], linestyle="--", lw=1.8
    )
    p_str = pfmt_extreme_p(p)
    print(f"{st}: r={r:.4f}, p={p_str}")

    # legend entries: marker + line for this state
    legend_handles.append(
        Line2D([0], [0], marker="o", linestyle="None",
               color=wake_palette[st], label=st)
    )
    legend_handles.append(
        Line2D([0], [0], color=wake_palette[st], linestyle="--", lw=1.8,
               label=f"Fit {st}, r={r:.2f}")
    )

ax.set_title("Task wake (T-maze)\nWake θ + Wake non-θ")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(handles=legend_handles, frameon=False, fontsize=12)

# --------------------------------
# RIGHT: sleep (NREM vs REM)
# --------------------------------
ax = axes[1]
legend_handles_sleep = []
pad_sleep={"NREM":0.3,"REM":0.05}
for st in ["NREM", "REM"]:
    sub = sleep_pairs[sleep_pairs["sleep_substate"] == st]
    if sub.empty:
        continue

    x = sub["mua30_z"].to_numpy(float)
    y = sub["d2_z"].to_numpy(float)
    x_min_state, x_max_state = local_limits(x,pad_frac=pad_sleep[st])
    # scatter points
    sc = ax.scatter(
        x, y,
        s=4, alpha=0.8,
        color=sleep_palette[st],
        edgecolors="none",
        rasterized=True
    )
    sc.set_rasterized(True)

    # fit line for THIS sleep substate only
    lr, r, p = add_fit_line(
        ax, x, y,
        x_min=x_min_state, x_max=x_max_state,
        color=sleep_inline_palette[st], linestyle="--", lw=1.8
    )
    print(f"{st}: r={r:.4f}, p={p:.3e}")

    # legend entries
    legend_handles_sleep.append(
        Line2D([0], [0], marker="o", linestyle="None",
               color=sleep_palette[st], label=st)
    )
    legend_handles_sleep.append(
        Line2D([0], [0], color=sleep_palette[st], linestyle="--", lw=1.8,
               label=f"Fit {st}, r={r:.2f}")
    )

ax.set_title("Sleep\nNREM + REM")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(handles=legend_handles_sleep, frameon=False, fontsize=12)


# shared labels
fig.supxlabel("Mean population firing rate (z)")  # MUA_z on x
fig.supylabel("d distance (z)")                           # d2_z on y

plt.tight_layout()
fig.savefig(r"../figure_output\figS1_D_MUA_d2_scatter.svg", format="svg", dpi=300, bbox_inches="tight")
plt.show()